# Web scraping ofert

In [12]:
import json
import urllib.parse
import requests
from bs4 import BeautifulSoup


def generate_search_url(filters: dict) -> str:
    f = filters.copy()

    path = ["pl"]
    if wm := f.pop("work_model", None):
        path.append(wm)

    techs = f.pop("technologies", [])
    techs = techs if isinstance(techs, list) else ([techs] if techs else [])

    t_map = {"C++": "C%2B%2B", "C#": "C%23"}
    techs = [t_map.get(t, str(t)) for t in techs]

    if techs:
        path.append(techs[0])
        if len(techs) > 1:
            f["requirement"] = techs[1:]

    url = "https://nofluffjobs.com/" + "/".join(path)

    criteria = []
    sr = f.pop("salary_range", [])
    if len(sr) == 2:
        criteria.extend([f"salary>pln{sr[0]}m", f"salary<pln{sr[1]}m"])

    sort_val = f.pop("sort", None)

    for k, v in f.items():
        val = ",".join(str(i) for i in v) if isinstance(v, list) else str(v)
        criteria.append(f"{k}={val}")

    query_params = []
    if criteria:
        qs = urllib.parse.quote(" ".join(criteria), safe=',%')
        query_params.append(f"criteria={qs}")

    if sort_val:
        query_params.append(f"sort={sort_val}")

    if query_params:
        url += "?" + "&".join(query_params)

    return url


def get_offers_from_search_url(search_url: str) -> list[str]:
    response = requests.get(search_url, headers={"User-Agent": "."})
    soup = BeautifulSoup(response.text, "lxml")

    if soup.find("nfj-no-offers-found-header"):
        return []

    all_slugs = []
    for item in soup.find_all("nfj-postings-list", {"data-cy-params": '{"search_results":"standard"}'}):
        all_links = item.find_all("a")
        for link in all_links:
            if link["href"].startswith("/pl/job"):
                all_slugs.append(link["href"])

    return [urllib.parse.urljoin("https://nofluffjobs.com", slug) for slug in all_slugs]


def scrape_single_offer(url: str) -> dict:
    response = requests.get(url, headers={"User-Agent": "."})
    soup = BeautifulSoup(response.text, "lxml")
    all_data = json.loads(soup.find("script", {"id": "serverApp-state"}).text)
    data_key = next((key for key in all_data.keys() if key.startswith("/posting/")), None)
    data = all_data[data_key]

    company_name = data.get("company", {}).get("name", "")
    company_size = data.get("company", {}).get("size", "")
    job_category = data.get("basics", {}).get("category", "")
    seniority = ", ".join(data.get("basics", {}).get("seniority", []))
    contract_start = data.get("essentials", {}).get("contract", {}).get("start", "")

    salary_data = data.get("essentials", {}).get("originalSalary", {})
    currency = salary_data.get("currency", "")
    salary_info = "\n".join([
        f"{t.upper()}: {info['range'][0]}-{info['range'][1]} {currency} / {info['period']}"
        for t, info in salary_data.get("types", {}).items()
        if "range" in info
    ])

    job_description = data.get("details", {}).get("description", "")
    job_location = " / ".join(
        [item.get("city", "") for item in data.get("location", {}).get("places", [])])
    must_have_skills = " | ".join(
        [skill.get("value", "") for skill in data.get("requirements", {}).get("musts", [])])
    nice_to_have_skills = " | ".join(
        [skill.get("value", "") for skill in data.get("requirements", {}).get("nices", [])])
    requirements_description = data.get("requirements", {}).get("description", "")

    return {
        "company_name": company_name,
        "company_size": company_size,
        "job_category": job_category,
        "seniority": seniority,
        "contract_start": contract_start,
        "salary_info": salary_info,
        "job_description": job_description,
        "job_location": job_location,
        "must_have_skills": must_have_skills,
        "nice_to_have_skills": nice_to_have_skills,
        "requirements_description": requirements_description,
    }

---

In [13]:
filters = {
    "work_model": "hybrid",  # hybrid / praca-zdalna
    "technologies": ["Python", "C++"], # Python / C++ / C# / Java / JavaScript / TypeScript
    "city": ["krakow", "warszawa"],  # krakow / warszawa / wroclaw / poznan / gdansk
    "employment": ["b2b", "permanent"],  # b2b / permanent
    "salary_range": [5000, 20000],  # <salary_min> / <salary_max>
    "seniority": ["senior"], # junior / mid / senior
    "sort": "newest"
}


search_url = generate_search_url(filters)
search_url

'https://nofluffjobs.com/pl/hybrid/Python?criteria=salary%3Epln5000m%20salary%3Cpln20000m%20city%3Dkrakow,warszawa%20employment%3Db2b,permanent%20seniority%3Dsenior%20requirement%3DC%2B%2B&sort=newest'

In [6]:
offer_urls = get_offers_from_search_url(search_url)
offer_urls

[]

In [8]:
scrape_single_offer("https://nofluffjobs.com/pl/job/konsultant-konsultantka-ds-obslugi-klienta-z-jezykiem-polskim-i-ukrainskim-teleperformance-poland-warsaw")

AttributeError: 'NoneType' object has no attribute 'text'

In [4]:
offers = [scrape_single_offer(url) for url in offer_urls]
offers

[]

In [5]:
offers[0]

IndexError: list index out of range